In [ ]:
!git config --global user.name "VishwaaShah5"
!git config --global user.email "vishwaashah2004@gmail.com"

In [ ]:
!git clone https://github.com/VishwaaShah5/DimABSA_SemEval.git

Cloning into 'DimABSA_SemEval'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.


In [ ]:
!cp Prompting_With_Mistral.ipynb DimABSA_SemEval/

cp: cannot stat 'Prompting_With_Mistral.ipynb': No such file or directory


Prompting with Mistral

**Approach:** Few-shot prompting with Mistral-7B-Instruct

**Workflow:**
1. Train on small_train (80%)
2. Validate on valid (20%) → Show RMSE
3. Retrain on small_train+valid (100%)
4. Predict on dev → Generate submission

**Target:** Validation RMSE < 1.5

## 1. Setup

In [ ]:
!pip install -q transformers accelerate bitsandbytes
from google.colab import drive
#drive.mount('/content/drive')
import json, re, numpy as np, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from tqdm import tqdm
print('✓ Libraries installed')

## 2. Load Mistral

In [ ]:
MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.2'

print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    torch_dtype=torch.bfloat16,
    device_map='auto'
)

pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=20,
    do_sample=True,
    temperature=0.1,
    pad_token_id=tokenizer.eos_token_id
)

print('✓ Mistral loaded in 4-bit')

## 3. Load Data

In [ ]:
def load_jsonl(fp):
    return [json.loads(l) for l in open(fp) if l.strip()]

train_rest = load_jsonl('/content/eng_restaurant_small_train.jsonl')
train_laptop = load_jsonl('/content/eng_laptop_small_train.jsonl')
valid_rest = load_jsonl('/content/eng_restaurant_valid.jsonl')
valid_laptop = load_jsonl('/content/eng_laptop_valid.jsonl')
test_rest = load_jsonl('/content/eng_restaurant_dev_task1.jsonl')
test_laptop = load_jsonl('/content/eng_laptop_dev_task1.jsonl')

print(f'Train: {len(train_rest)+len(train_laptop)} instances')
print(f'Valid: {len(valid_rest)+len(valid_laptop)} instances')
print(f'Test: {len(test_rest)+len(test_laptop)} instances')

In [ ]:
def extract_pairs(data, domain):
    pairs = []
    for item in data:
        for quad in item['Quadruplet']:
            aspect = quad['Aspect'] if quad['Aspect'] != 'NULL' else quad['Category']
            pairs.append({'text': item['Text'], 'aspect': aspect, 'va': quad['VA'], 'domain': domain})
    return pairs

train_rest_pairs = extract_pairs(train_rest, 'restaurant')
train_laptop_pairs = extract_pairs(train_laptop, 'laptop')
print(f'Training pairs: {len(train_rest_pairs) + len(train_laptop_pairs)}')

## 4. Few-Shot Prompting

In [ ]:
import random
random.seed(42)
rest_ex = random.sample(train_rest_pairs, 10)
laptop_ex = random.sample(train_laptop_pairs, 10)

def create_prompt(text, aspect, domain, examples):
    p = f'''Predict Valence-Arousal for {domain} reviews.
Valence: 1.00 (negative) to 9.00 (positive)
Arousal: 1.00 (calm) to 9.00 (excited)
Format: V#A\n\n'''
    for ex in examples:
        p += f'Text: "{ex["text"]}"\nAspect: {ex["aspect"]}\nVA: {ex["va"]}\n\n'
    p += f'Text: "{text}"\nAspect: {aspect}\nVA: '
    return p

def parse_va(text):
    m = re.search(r'(\d+\.\d{2})#(\d+\.\d{2})', text)
    if m:
        return max(1.0, min(9.0, float(m.group(1)))), max(1.0, min(9.0, float(m.group(2))))
    return 5.0, 5.0

def predict_va(text, aspect, domain, examples):
    prompt = create_prompt(text, aspect, domain, examples)
    output = pipe(prompt)
    response = output[0]['generated_text'][len(prompt):]
    return parse_va(response)

## 5. Validate

In [ ]:
def evaluate_rmse(data, domain, examples):
    errors = []
    for item in tqdm(data, desc=domain):
        for quad in item['Quadruplet']:
            aspect = quad['Aspect'] if quad['Aspect'] != 'NULL' else quad['Category']
            gold_v, gold_a = map(float, quad['VA'].split('#'))
            pred_v, pred_a = predict_va(item['Text'], aspect, domain, examples)
            errors.append((pred_v - gold_v)**2 + (pred_a - gold_a)**2)
    return np.sqrt(np.mean(errors))

print('Validation RMSE (20 instances):')
rest_rmse = evaluate_rmse(valid_rest[:20], 'restaurant', rest_ex)
laptop_rmse = evaluate_rmse(valid_laptop[:20], 'laptop', laptop_ex)
avg_rmse = (rest_rmse + laptop_rmse) / 2
print(f'\nRestaurant: {rest_rmse:.3f}')
print(f'Laptop: {laptop_rmse:.3f}')
print(f'Average: {avg_rmse:.3f}')
print(f'Target: < 1.5')

## 6. Retrain on Full Data

In [ ]:
full_rest_pairs = extract_pairs(train_rest + valid_rest, 'restaurant')
full_laptop_pairs = extract_pairs(train_laptop + valid_laptop, 'laptop')
final_rest_ex = random.sample(full_rest_pairs, 10)
final_laptop_ex = random.sample(full_laptop_pairs, 10)
print(f'Final training: {len(full_rest_pairs) + len(full_laptop_pairs)} pairs')

## 7. Generate Test Predictions

In [ ]:
def predict_test_set(test_data, domain, examples):
    predictions = []
    for item in tqdm(test_data, desc=domain):
        aspect_va_list = []
        for aspect in item['Aspect']:
            v, a = predict_va(item['Text'], aspect, domain, examples)
            aspect_va_list.append({'Aspect': aspect, 'VA': f'{v:.2f}#{a:.2f}'})
        predictions.append({'ID': item['ID'], 'Aspect_VA': aspect_va_list})
    return predictions

rest_preds = predict_test_set(test_rest, 'restaurant', final_rest_ex)
laptop_preds = predict_test_set(test_laptop, 'laptop', final_laptop_ex)

## 8. Save & Download

In [ ]:
def save(preds, path):
    with open(path, 'w') as f:
        for p in preds:
            f.write(json.dumps(p) + '\n')
    print(f'✓ Saved {len(preds)} to {path}')

save(rest_preds, '/content/pred_eng_restaurant.jsonl')
save(laptop_preds, '/content/pred_eng_laptop.jsonl')

from google.colab import files
files.download('/content/pred_eng_restaurant.jsonl')
files.download('/content/pred_eng_laptop.jsonl')